# 06 — API Inference and Production Simulation

**Objective:** Exercise health, readiness, validation, feature engineering, prediction, and repeated raw-row requests without starting an external server.

In [1]:
from pathlib import Path
import os, sys
import pandas as pd
import plotly.express as px

_cwd = Path.cwd().resolve()
ROOT = _cwd.parent if _cwd.name == "dev" else _cwd
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/credit-risk-lab-matplotlib")

from credit_risk_lab.config.settings import settings
print(f"Project root: {settings.project_root}")

Project root: /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab


## 1. Start the FastAPI application in-process

`TestClient` runs the same application used by Uvicorn and Docker.

In [2]:
from fastapi.testclient import TestClient
from credit_risk_lab.interfaces.api import app

client = TestClient(app)
client.__enter__()
{"health": client.get("/health").json(), "ready": client.get("/ready").json()}

{'health': {'status': 'ok', 'service': 'Credit Risk Lab'},
 'ready': {'status': 'ready', 'model': 'LightGBM'}}

## 2. Send one raw application

The label is removed before the request. The API validates input, engineers features, scores, and returns audit metadata.

In [3]:
from credit_risk_lab.infrastructure.data_sources import CSVDataSourceConfig, CSVDatasetRepository

test_df = CSVDatasetRepository(CSVDataSourceConfig(path=settings.test_path)).load()
row = test_df.iloc[0]
payload = row.drop(labels=[settings.target_column]).to_dict()
response = client.post("/v1/predict", json=payload, headers={"X-Request-ID": "notebook-example-1"})
response.status_code, response.json(), {"expected_label": int(row[settings.target_column])}

2026-07-11 09:52:42 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:51 - Chargement du fichier : /Users/surelmanda/Mlops-Databricks-Projects/credit-risk-lab/data/processed/test.csv


2026-07-11 09:52:42 | INFO     | CSVDatasetRepository | credit_risk_lab.infrastructure.data_sources.csv_dataset_repository:load:59 - Dataset chargé (4500 lignes, 14 colonnes)


2026-07-11 09:52:42 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Entrée - shape = (1, 13)


2026-07-11 09:52:42 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Sortie - shape = (1, 38)


(200,
 {'request_id': 'notebook-example-1',
  'model_name': 'LightGBM',
  'model_version': '0.2.0',
  'probability_of_risk': 0.15599377947899687,
  'risk_decision': 0,
  'risk_label': 'low_risk',
  'threshold': 0.25,
  'threshold_source': 'configs/settings.yaml:decision_threshold',
  'scored_at_utc': '2026-07-11T07:52:42.461172Z',
  'latency_ms': 32.523},
 {'expected_label': 0})

## 3. Simulate a short production stream

In [4]:
responses = []
for index, row in test_df.head(20).iterrows():
    payload = row.drop(labels=[settings.target_column]).to_dict()
    http = client.post("/v1/predict", json=payload)
    body = http.json()
    responses.append({
        "row": int(index), "http_status": http.status_code,
        "expected": int(row[settings.target_column]),
        "probability": body["probability_of_risk"],
        "decision": body["risk_decision"], "latency_ms": body["latency_ms"],
    })
simulation = pd.DataFrame(responses)
simulation

2026-07-11 09:52:42 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Entrée - shape = (1, 13)


2026-07-11 09:52:42 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Sortie - shape = (1, 38)


2026-07-11 09:52:42 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Entrée - shape = (1, 13)


2026-07-11 09:52:42 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Sortie - shape = (1, 38)


2026-07-11 09:52:42 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Entrée - shape = (1, 13)


...

2026-07-11 09:52:42 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Entrée - shape = (1, 13)


2026-07-11 09:52:42 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Sortie - shape = (1, 38)


2026-07-11 09:52:42 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Entrée - shape = (1, 13)


2026-07-11 09:52:42 | INFO     | api_features | credit_risk_lab.infrastructure.feature_engineering.pandas_feature_engineer:_log_shape:126 - [FeatureEngineering] Sortie - shape = (1, 38)


,row,http_status,expected,probability,decision,latency_ms
0,0,200,0,0.155994,0,20.249
1,1,200,0,0.000165,0,20.883
2,2,200,0,0.066986,0,20.029
3,3,200,0,0.013961,0,19.816
4,4,200,0,0.000165,0,21.126
5,5,200,0,0.000174,0,19.397
6,6,200,1,0.303449,1,19.827
7,7,200,0,0.031949,0,19.676
8,8,200,0,0.150012,0,19.698
9,9,200,1,0.933342,1,18.773


## 4. Inspect simulation behavior

In [5]:
display(simulation.describe())
px.histogram(simulation, x="probability", color="expected", nbins=20, title="Simulated API risk probabilities").show()

,row,http_status,expected,probability,decision,latency_ms
count,20.00000,20.0,20.000000,20.000000,20.000000,20.00000
mean,9.50000,200.0,0.150000,0.147439,0.150000,19.42945
std,5.91608,0.0,0.366348,0.292130,0.366348,0.80698
min,0.00000,200.0,0.000000,0.000165,0.000000,18.18700
25%,4.75000,200.0,0.000000,0.000165,0.000000,18.76575
50%,9.50000,200.0,0.000000,0.007068,0.000000,19.61600
75%,14.25000,200.0,0.000000,0.151508,0.000000,19.81875
max,19.00000,200.0,1.000000,0.995190,1.000000,21.12600


## 5. Validate a bad request and close the client

In [6]:
invalid = payload.copy()
invalid["person_age"] = 20
invalid["person_emp_exp"] = 10
invalid_response = client.post("/v1/predict", json=invalid)
client.__exit__(None, None, None)
{"status": invalid_response.status_code, "detail": invalid_response.json()}

{'status': 422,
 'detail': {'detail': [{'type': 'value_error',
    'loc': ['body'],
    'msg': 'Value error, Employment experience is implausible for age',
    'input': {'person_age': 20,
     'person_gender': 'male',
     'person_education': 'High School',
     'person_income': 57803.0,
     'person_emp_exp': 10,
     'person_home_ownership': 'MORTGAGE',
     'loan_amnt': 14000.0,
     'loan_intent': 'EDUCATION',
     'loan_int_rate': 6.03,
     'loan_percent_income': 0.24,
     'cb_person_cred_hist_length': 2.0,
     'credit_score': 598,
     'previous_loan_defaults_on_file': 'Yes'},
    'ctx': {'error': {}}}]}}